# Translation model comparison - Qwen3-8B vs TranslateGemma-12B (4-bit)

Standalone benchmark for the **translation** step only: text in -> text out.
No ASR, no LiveKit, no TTS. It loads each translation model **one at a time**
on the GPU (frees VRAM between them), translates a set of Arabic sentences and
reports:

- **Latency** (seconds per sentence), and
- **Decode throughput** (generated tokens/s),

plus a side-by-side view of the actual translations so you can judge quality.

At the end, pick the winner and wire it into `bot.py` by setting
`TRANSLATION_BACKEND` in `.env`:

- `translategemma` -> `google/translategemma-12b-it`
- `qwen3` -> `Qwen/Qwen3-8B`

### Notes
- Run on a **GPU** (Colab: Runtime -> Change runtime type -> GPU). A 12B 4-bit
  model needs a sizeable GPU (e.g. ~16-24 GB). If you OOM, edit
  `MODELS_TO_TEST` below to test one model at a time.
- **TranslateGemma is gated**: accept the Gemma terms on Hugging Face and set
  an `HF_TOKEN`. Qwen3 is not gated.
- Both models run in **4-bit (NF4)** to mirror what `bot.py` uses at runtime.


In [ ]:
# Install dependencies if your environment is missing them
# (Colab GPU images usually already have torch + transformers).
# %pip install -q --upgrade torch "transformers>=5.4.0" bitsandbytes accelerate huggingface_hub
print("If the next cells import cleanly, your environment is ready.")


### Hugging Face token

`google/translategemma-12b-it` is a **gated** Gemma repo. Accept its license on
[huggingface.co/google/translategemma-12b-it](https://huggingface.co/google/translategemma-12b-it),
then set `HF_TOKEN` in the next cell (or in the environment / a Colab secret).
`qwen3` works without a token.


In [ ]:
import os

# ---- Configuration ----
# Which models to evaluate (any subset): "translategemma" and/or "qwen3".
MODELS_TO_TEST = ["translategemma", "qwen3"]

# Translation direction (ISO 639-1). The app translates Arabic -> German.
SOURCE_LANG = "ar"
TARGET_LANG = "de"

MAX_NEW_TOKENS = 256
WARMUP_RUNS = 1    # untimed runs per sentence (warms kernels/allocations)
TIMED_RUNS = 3     # timed runs per sentence (averaged)

# ---- Auth (gated Google repo) ----
# Set your token here, or export HF_TOKEN in the environment first.
os.environ.setdefault("HF_TOKEN", "")
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip() or None
print("HF_TOKEN present:", bool(HF_TOKEN))


In [ ]:
import gc
import time

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForImageTextToText,
    AutoProcessor,
    AutoTokenizer,
    BitsAndBytesConfig,
)

QWEN_TRANSLATION_MODEL = "Qwen/Qwen3-8B"
GEMMA_TRANSLATION_MODEL = "google/translategemma-12b-it"


def pick_dtype_and_device():
    if not torch.cuda.is_available():
        print("WARNING: no CUDA GPU found; inference will be very slow.")
        return torch.float32, "cpu"
    major, _ = torch.cuda.get_device_capability(0)
    # bf16 on Ada+ (sm80+), otherwise fp16.
    return (torch.bfloat16 if major >= 8 else torch.float16), "cuda"


def bnb_4bit(dtype):
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


DTYPE, DEVICE = pick_dtype_and_device()
print("CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("compute dtype / device:", DTYPE, DEVICE)


In [ ]:
def load_backend(name, dtype, device):
    # Loads one translator (4-bit) and returns (model, encoder).
    token = {"token": HF_TOKEN} if HF_TOKEN else {}
    if name == "qwen3":
        model_id = QWEN_TRANSLATION_MODEL
        tokenizer = AutoTokenizer.from_pretrained(model_id, **token)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=dtype,
            quantization_config=bnb_4bit(dtype),
            device_map=device,
            **token,
        )
        model.eval()
        return model, tokenizer

    if name == "translategemma":
        model_id = GEMMA_TRANSLATION_MODEL
        processor = AutoProcessor.from_pretrained(model_id, **token)
        model = AutoModelForImageTextToText.from_pretrained(
            model_id,
            quantization_config=bnb_4bit(dtype),
            device_map=device,
            **token,
        )
        model.eval()
        return model, processor

    raise ValueError(f"Unknown backend {name!r} (use 'translategemma' or 'qwen3').")


STATE = {"name": None, "model": None, "encoder": None}


def unload():
    # Fully releases the loaded model and VRAM so the next one fits.
    STATE["name"] = STATE["model"] = STATE["encoder"] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def ensure_loaded(name, dtype, device):
    # Loads `name` (only if not already loaded). Returns (name, model, encoder).
    if STATE["name"] != name:
        unload()
        STATE["model"], STATE["encoder"] = load_backend(name, dtype, device)
        STATE["name"] = name
        print(f"Loaded '{name}'.")
    return STATE["name"], STATE["model"], STATE["encoder"]


In [ ]:
def translate_one(name, model, encoder, text):
    # Translate `text`. Returns (translation, prompt_tokens, generated_tokens).
    # Mirrors the exact inference each backend uses inside `bot.py`.
    if name == "qwen3":
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a professional Arabic-to-German translator. "
                    "Translate the current Arabic segment completely and "
                    "faithfully into natural, spoken German. Do not invent "
                    "anything, do not summarize, and output only the German "
                    "translation."
                ),
            },
            {"role": "user", "content": "Current Arabic segment:\n" + text + "\n\nGerman translation:"},
        ]
        prompt = encoder.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        inputs = encoder(prompt, return_tensors="pt").to(model.device)
        prompt_len = inputs["input_ids"].shape[1]
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.05)
        new = out[0, prompt_len:]
        return encoder.decode(new, skip_special_tokens=True).strip(), prompt_len, new.shape[0]

    # translategemma: fixed chat template (language codes + text), no context.
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "source_lang_code": SOURCE_LANG,
                    "target_lang_code": TARGET_LANG,
                    "text": text,
                }
            ],
        }
    ]
    inputs = encoder.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new = out[0, prompt_len:]
    return encoder.decode(new, skip_special_tokens=True).strip(), prompt_len, new.shape[0]


def bench_backend(name, model, encoder, sentences):
    # Warm-up (untimed) so kernels and allocations settle.
    for _ in range(WARMUP_RUNS):
        for s in sentences:
            translate_one(name, model, encoder, s)
    _sync()

    rows = []
    for s in sentences:
        times, translation, prompt_len, new_tokens = [], "", 0, 0
        for _ in range(TIMED_RUNS):
            _sync()
            t0 = time.perf_counter()
            translation, prompt_len, new_tokens = translate_one(name, model, encoder, s)
            _sync()
            times.append(time.perf_counter() - t0)
        avg = sum(times) / len(times)
        rows.append(
            {
                "text": s,
                "translation": translation,
                "avg_s": avg,
                "min_s": min(times),
                "prompt_tokens": prompt_len,
                "gen_tokens": new_tokens,
                "tok_per_s": new_tokens / avg,
            }
        )
    return rows


In [ ]:
# Sample Arabic sentences (as if transcribed from speech). The German
# "reference" is for judging quality only - the models never see it.
SENTENCES = [
    ("مرحباً بكم في حلقة جديدة من البرنامج", "Willkommen zu einer neuen Folge der Sendung"),
    ("كيف حالك اليوم؟", "Wie geht es dir heute?"),
    ("أريد أن أطلب قهوة وحلويات", "Ich möchte Kaffee und Süßigkeiten bestellen"),
    ("المسافة من بيتي إلى العمل ساعة تقريبا", "Der Weg von meinem Haus zur Arbeit dauert etwa eine Stunde"),
    ("الطقس اليوم حار جدا", "Das Wetter ist heute sehr heiß"),
    ("سأتصل بك لاحقا لأن لدي اجتماعا الآن", "Ich rufe dich später an, weil ich gerade ein Meeting habe"),
]
SRC = [s for s, _ in SENTENCES]
print(f"{len(SENTENCES)} test sentences ready ({SOURCE_LANG} -> {TARGET_LANG}).")


### Run the benchmark

Loads each selected model one at a time, warms it up, then times `TIMED_RUNS`
translations of every sentence.


In [ ]:
results = {}
for name in MODELS_TO_TEST:
    print(f"\n=== {name} ===")
    n, model, encoder = ensure_loaded(name, DTYPE, DEVICE)
    results[name] = bench_backend(name, model, encoder, SRC)
    if torch.cuda.is_available():
        print(f"peak VRAM for '{name}': {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
unload()
print("\nBenchmark complete.")


In [ ]:
# Performance summary (lower latency / higher tok-s is faster).
header = f"{'model':<15} {'avg s/sentence':>16} {'min s/sentence':>16} {'avg tok/s':>12}"
print(header)
print("-" * len(header))
for name, rows in results.items():
    avg = sum(r["avg_s"] for r in rows) / len(rows)
    mn = min(r["min_s"] for r in rows)
    tps = sum(r["tok_per_s"] for r in rows) / len(rows)
    print(f"{name:<15} {avg:>16.3f} {mn:>16.3f} {tps:>12.1f}")


In [ ]:
# Quality: side-by-side translations for each input.
names = list(results)
w = 40  # per-column width (approx.)


def _clip(s, n):
    s = str(s)
    return (s[: (n - 1)] + "…") if len(s) > n else s.ljust(n)


header = f"{'#':>2}  {SOURCE_LANG + ' (input)':<{w}} " + " ".join(f"{n:<{w}}" for n in names) + f" {'reference (DE)':<{w}}"
print(header)
print("-" * (4 + w * (len(names) + 2)))
for i, (src, ref) in enumerate(SENTENCES):
    cells = [_clip(results[n][i]["translation"], w) for n in names]
    print(f"{i + 1:>2}  " + _clip(src, w) + " " + " ".join(cells) + " " + _clip(ref, w))


### Playground - translate your own text

Enter any Arabic sentence and translate it with whichever model you like; the
chosen model is loaded on demand.


In [ ]:
def translate_arbitrary(text, name):
    n, model, encoder = ensure_loaded(name, DTYPE, DEVICE)
    _sync()
    t0 = time.perf_counter()
    translation, prompt_len, new_tokens = translate_one(n, model, encoder, text)
    _sync()
    return translation, time.perf_counter() - t0, new_tokens

# Edit this to your own sentence, then run:
demo_text = "شكرا جزيلا لكم على استضافتي في هذا البرنامج الرائع"
for name in MODELS_TO_TEST:
    tr, dt, nt = translate_arbitrary(demo_text, name)
    print(f"[{name}] {nt} new tokens in {dt:.2f}s")
    print("   ", tr)
    print()
unload()


### Keeping your winner in `bot.py`

Set `TRANSLATION_BACKEND` in the repo `.env`:

- `TRANSLATION_BACKEND=translategemma` -> `google/translategemma-12b-it`
- `TRANSLATION_BACKEND=qwen3` -> `Qwen/Qwen3-8B`

Notes:
- `bot.py` always loads the translator in **4-bit (NF4)** and picks bf16/fp16
  compute from the GPU - same as this notebook.
- The **qwen3** path also sends the last few segments as *conversation
  context* to resolve references; **translategemma**'s fixed chat template has
  no context (one segment at a time).
- If you choose `translategemma`, set `HF_TOKEN` in `.env` (gated repo).
- Direction defaults to `ar -> de`; override with `TRANSLATION_SOURCE_LANG` /
  `TRANSLATION_TARGET_LANG` in `.env`.
